# 多列布局与打印分页

学习目标：能让连续文章按列流动，并为同一份内容设置打印样式、页边距和分片边界。

前置知识：正常流、盒模型、长度单位、媒体查询与浏览器打印预览操作。

适用范围：CSS Multi-column Layout Level 1、Fragmentation Level 3 与 Paged Media Level 3 的相关功能；这些模块仍按各自规范状态演进。column-fill 和高级分页功能存在实现差异，以目标浏览器打印预览为准。

环境准备：[环境配置与运行](README.md)。

配套脚本：位于 scripts/23-multicol-and-print/。

1. [index.html](scripts/23-multicol-and-print/index.html)：列宽、跨列、填列和溢出。
2. [print.html](scripts/23-multicol-and-print/print.html)：长讲义与强制新页附录。
3. [styles.css](scripts/23-multicol-and-print/styles.css)：屏幕与打印规则。

Step 1：在已激活 Python 环境的终端中，从项目根目录进入本技术目录。

```bash
cd content/Web与应用开发/css
```

Step 2：启动本章预览服务。

```bash
python -m http.server 8101 --bind 127.0.0.1
```

Step 3：打开[本章示例首页](http://127.0.0.1:8101/scripts/23-multicol-and-print/index.html)。

保存修改后刷新页面。

Step 4：在服务终端按 Ctrl+C 停止服务。

## 1 让一篇文章流过多列

多列布局（multi-column layout）把一份正常流内容分成多个列片段：先沿块方向读完一列，再进入下一列。本例为水平书写、从左到右排列；它适合连续文章，不承担 Grid 那种独立行列单元的对齐任务。

columns 是 column-width 与 column-count 的简写。18rem 是期望列宽，3 是列数上限；可用空间还要扣掉列间距。实际列宽可以分得更多空间；窄到容不下 18rem 时，唯一的一列也可比它更窄。

只指定 column-count 时按列数分配宽度，只指定 column-width 时自动决定能容纳多少列。两者一起写，并不保证始终显示三列。column-rule 画在列间隙中，不额外占据布局宽度。

```html
<article class="news">
  <h2 class="span">工作坊通讯</h2>
  <p>（1）先按原始顺序组织文字。多列排版让连续内容从一列流向下一列；列本身不是额外的 HTML 元素。</p>
  <p>（2）调整窗口宽度，观察相同内容怎样重新分片。列数减少时，文字仍然完整保留，不需要复制成另一套内容。</p>
  <h3 class="span">下午的安排</h3>
  <p>（3）跨列标题将前后的文字分成两组。先读完标题前各列，再读标题和后面的内容。</p>
  <p>（4）每个段落都保留清楚的文字序号。不要把按行比较的产品表格改成这种连续文章布局。</p>
</article>
```

```css
.news {
  columns: 18rem 3;
  column-gap: 2rem;
  column-rule: 1px solid #6b7280;
}
/* 18rem 是期望列宽，3 是列数上限；空间不足时可以只剩一列。 */
```

配套文件：[index.html](scripts/23-multicol-and-print/index.html)、[styles.css](scripts/23-multicol-and-print/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/23-multicol-and-print/index.html)

## 2 本章使用的属性

| 完整属性名 | 中文名称／含义 | 用途或对象 |
| --- | --- | --- |
| columns | 列设置简写 | 同时设置列宽与列数 |
| column-width | 期望列宽 | 多列容器 |
| column-count | 列数 | 单独指定列数，或与列宽配合限定上限 |
| column-gap | 列间距 | 多列容器列与列之间 |
| column-rule | 列分隔线简写 | 绘制列间分隔线 |
| column-span | 跨列方式 | 正常流中的跨列块 |
| column-fill | 列填充方式 | 均衡或按顺序填充 |
| break-before | 元素前的分片控制 | 强制换列、换页或避免分片 |
| break-after | 元素后的分片控制 | 避免标题后直接换页 |
| break-inside | 元素内部的分片控制 | 尽量保持短内容组完整 |
| orphans | 分片前最少行数 | 段落留在前一片段的行数 |
| widows | 分片后最少行数 | 段落留在后一片段的行数 |

all、balance、auto、avoid、column 和 page 是对应属性的值。@media 与 @page 是 @ 规则；@page 中的页面设置在第 6 节单独说明。

## 3 跨列标题与填列策略

column-span: all 让正常流中的块跨越当前多列容器的全部列，不接受“跨两列”这样的数字。跨列块之前的内容会先进行列平衡，之后开始新的列组；跨列块自身建立块级格式化上下文。

示例把跨列标题直接放在 .news 中，避免中间的独立格式化上下文隔断跨列关系。不要把它和 grid-column 混用来理解。

column-fill: balance 是默认均衡策略；auto 按顺序填满一列再去下一列。本例为两组相同短文字指定 12rem 高度，再比较填充。没有确定块方向尺寸时，各浏览器对 auto 的处理存在差异；在分页媒体中，balance 通常只平衡最后一个片段，而不是承诺每一页都相同。

```html
<h2>均衡填列</h2>
<div class="fill balanced"><p>第 1 条短记录。</p>
<p>第 2 条短记录。</p>
<p>第 3 条短记录。</p>
<p>第 4 条短记录。</p>
<p>第 5 条短记录。</p>
<p>第 6 条短记录。</p></div>
<h2>依次填列</h2>
<div class="fill sequential"><p>第 1 条短记录。</p>
<p>第 2 条短记录。</p>
<p>第 3 条短记录。</p>
<p>第 4 条短记录。</p>
<p>第 5 条短记录。</p>
<p>第 6 条短记录。</p></div>
```

```css
.span { column-span: all; }
/* 跨越当前多列容器的全部列；观察中间标题前后的两组文字。 */

.fill { columns: 3; block-size: 12rem; column-gap: 1rem; line-height: 1.5; }
.fill p { margin: 0; }
.balanced { column-fill: balance; }
.sequential { column-fill: auto; }
/* 两组有相同六条文字与高度；比较最后一条记录位于哪一列。 */
```

配套文件：[index.html](scripts/23-multicol-and-print/index.html)、[styles.css](scripts/23-multicol-and-print/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/23-multicol-and-print/index.html)

## 4 分片控制与放不下的内容

分片（fragmentation）指同一内容分到不同列或页。break-before 和 break-after 管元素边界，break-inside 管元素内部；它们需要处于相应的分片上下文才有意义。

- column 强制换到下一列；page 强制换到下一页，不能用换列值代替换页值。
- avoid 尽量避免相应位置的分片；avoid-column 和 avoid-page 可以只限制一种分片。
- 强制分片优先于避免分片。同一边界有多个强制要求时，较靠后的 break-before 优先于前一元素的 break-after。

避免拆分不是不可违反的承诺：如果整个块比一页或一列还高，浏览器可能放宽约束来继续排版。因此把短说明组设为 avoid 可以，把整篇长讲义都设为 avoid 往往无益。

```html
<div class="break-demo">
  <section class="keep"><h3>一组记录</h3><p>标题与这段短文字尽量留在同一列。</p></section>
  <section class="new-column"><h3>另起一列</h3><p>即使前列尚有空间，也从下一列开始。</p></section>
</div>
```

```css
.break-demo { columns: 2; column-gap: 2rem; }
.keep { break-inside: avoid; }
.new-column { break-before: column; }
/* avoid 尽量避免拆开内容，column 则明确要求换列。 */
```

配套文件：[index.html](scripts/23-multicol-and-print/index.html)、[styles.css](scripts/23-multicol-and-print/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/23-multicol-and-print/index.html)

## 5 固定列高带来的横向溢出

屏幕是连续媒体。固定多列容器的块方向尺寸后，内容可能在行内方向继续生成溢出列；不能假设 column-count 会让多余内容消失。

这里故意限制高度，以可聚焦的外层滚动区域保留全部文字；它是观察边界用的例子。长文章通常优先让高度自然增长，避免读完一列后反复上下滚动或横向找后续内容。

tabindex="0" 让这个滚动区域进入键盘顺序，aria-label 为它命名。聚焦后用方向键检查末尾的第 12 条记录能否到达；滚动条外观可能由系统控制，不能仅凭是否看见滚动条判断没有溢出。

```html
<div class="overflow-window" tabindex="0" role="region" aria-label="固定高度多列溢出演示"><div class="overflow-columns"><p>记录 1：固定高度装不下时，继续向行内方向产生列。请横向滚动阅读后续内容。</p>
<p>记录 2：固定高度装不下时，继续向行内方向产生列。请横向滚动阅读后续内容。</p>
<p>记录 3：固定高度装不下时，继续向行内方向产生列。请横向滚动阅读后续内容。</p>
<p>记录 4：固定高度装不下时，继续向行内方向产生列。请横向滚动阅读后续内容。</p>
<p>记录 5：固定高度装不下时，继续向行内方向产生列。请横向滚动阅读后续内容。</p>
<p>记录 6：固定高度装不下时，继续向行内方向产生列。请横向滚动阅读后续内容。</p>
<p>记录 7：固定高度装不下时，继续向行内方向产生列。请横向滚动阅读后续内容。</p>
<p>记录 8：固定高度装不下时，继续向行内方向产生列。请横向滚动阅读后续内容。</p>
<p>记录 9：固定高度装不下时，继续向行内方向产生列。请横向滚动阅读后续内容。</p>
<p>记录 10：固定高度装不下时，继续向行内方向产生列。请横向滚动阅读后续内容。</p>
<p>记录 11：固定高度装不下时，继续向行内方向产生列。请横向滚动阅读后续内容。</p>
<p>记录 12：固定高度装不下时，继续向行内方向产生列。请横向滚动阅读后续内容。</p></div></div>
```

```css
.overflow-window { overflow-x: auto; border: 1px solid #6b7280; }
.overflow-columns {
  columns: 14rem 2;
  block-size: 10rem;
  column-fill: auto;
  column-gap: 2rem;
}
/* 此处特意限制高度，用外层横向滚动展示新增列，不把剩余文字裁掉。 */
```

配套文件：[index.html](scripts/23-multicol-and-print/index.html)、[styles.css](scripts/23-multicol-and-print/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/23-multicol-and-print/index.html)

## 6 打印媒体、页盒与页边距

@media print 选择打印媒体中的样式，仍服从通常的层叠与优先级。它改变页面内容的呈现；@page 则配置输出的页盒（page box），不是选择 HTML 中某个“页面元素”。

@page 中，size 是纸张尺寸与方向描述符；A4 portrait 表示目标为 A4 纵向。margin 设置页盒边距，margin-top 设置上边距；这些是页上下文设置，不放入普通元素的属性总览表。:first 是页面选择条件，选择首张页面，不是 DOM 的 :first-child。

本例页边距使用 mm；不要把 body 的 margin 当作纸张页边距。用户纸张、缩放、打印机可打印区域和页眉页脚选项都可能改变最终输出。@page 的所有扩展能力并非各浏览器都实现，本例只依赖基本尺寸和边距。

```html
<nav class="screen-only"><a href="index.html">返回屏幕示例</a></nav>
```

```css
@page {
  size: A4 portrait;
  margin: 18mm;
}
@page :first { margin-top: 24mm; }
/* 页盒设置不同于 body 的外边距；打印机与用户设置仍可能影响最终输出。 */
```

配套文件：[print.html](scripts/23-multicol-and-print/print.html)、[styles.css](scripts/23-multicol-and-print/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/23-multicol-and-print/print.html)

## 7 从屏幕布局转为分页输出

打印讲义在屏幕上按列阅读，打印时改为普通单列，并去掉屏幕导航。纸面没有可供读者拖动的滚动容器，受限高度与裁剪必须重新评估；本例显式恢复自然高度和可见溢出。

标题后使用 break-after: avoid-page，短说明用 break-inside: avoid，附录用 break-before: page。orphans: 3 要求被拆开的段落在前一片段至少留三行，widows: 3 要求后一片段至少留三行；它们按行盒控制，不能用来绑定两个独立段落，而且约束可能因空间不足被放宽。

打开 print.html 的浏览器打印预览，选择 A4、100% 缩放，逐页检查：从首行到末行是否完整、附录是否另起新页、短说明是否分开、页边距是否合适。再改变纸张或缩放重新检查。

开发者工具仿真 print 媒体只能帮助检查样式匹配，不能代替真实分页预览。屏幕截图、计算样式和样式解析通过，都不能证明最终分页正确。

```html
<section class="appendix">
  <h2>附录：纸面检查单</h2>
  <p>检查标题、段落末行、页边距，以及是否有被截断的内容。</p>
</section>
```

```css
@media print {
  .screen-only { display: none; }
  body { margin: 0; padding: 0; max-width: none; color: #000; background: #fff; }
  .report { columns: auto; block-size: auto; overflow: visible; }
  h2, h3 { break-after: avoid-page; }
  .keep { break-inside: avoid; }
  .report p { orphans: 3; widows: 3; }
  .appendix { break-before: page; }
  /* 在真实打印预览中检查：导航不打印，正文单列，附录从新页开始。 */
}
```

配套文件：[print.html](scripts/23-multicol-and-print/print.html)、[styles.css](scripts/23-multicol-and-print/styles.css) · [浏览器预览](http://127.0.0.1:8101/scripts/23-multicol-and-print/print.html)

## 本章小结

- 多列分割一份连续内容；列宽是期望值，列数与可用空间共同决定排版。
- 跨列标题会分开列组；填列策略需要连同高度和媒体类型观察。
- 强制分片、避免分片和行数约束作用不同，空间不足时不保证全部约束都成立。
- 打印要同时检查内容样式、页盒设置与实际分页预览。

## 练习

（1）把通讯的期望列宽改为 22rem，保持列数上限为 3。标准：在窄屏只剩一列，宽屏不超过三列，全文顺序不变。

（2）交换两组短记录的 column-fill 值，随后移除块方向尺寸。标准：记录相同高度下填列差异，并区分移除高度后的目标浏览器行为，不把它概括成跨浏览器结论。

（3）在打印讲义中增加一个超过整页高度的说明组，仍设置 break-inside: avoid。标准：在真实打印预览中确认内容没有丢失，说明为什么 avoid 不保证整组留在一页；再恢复示例。

### 提示

列数不能只看计算样式里的 column-count 判断，需观察实际列片段。打印检查同时记录纸张、缩放、页眉页脚开关和浏览器版本。

## 参考与引用来源

- W3C：[CSS Multi-column Layout Level 1 §3–7](https://www.w3.org/TR/css-multicol-1/#the-number-and-width-of-columns) 的列宽、列数、间距、跨列和填列；[CSS Fragmentation Level 3 §3–4](https://www.w3.org/TR/css-break-3/#breaking-rules) 的强制／非强制分片与约束放宽，[§3.3](https://www.w3.org/TR/css-break-3/#widows-orphans) 的 orphans、widows；[CSS Paged Media Level 3](https://www.w3.org/TR/css-page-3/#page-size) 的页面尺寸与页盒。
- MDN：[Basic concepts of multi-column layout](https://developer.mozilla.org/en-US/docs/Web/CSS/CSS_multicol_layout/Basic_concepts) 的列宽与列数配合；[column-span](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/column-span#values) 的跨列和平衡；[column-fill 的 Browser compatibility](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/column-fill#browser_compatibility) 的确定高度与实现差异；[break-inside](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/break-inside#syntax) 的边界冲突；[Handling overflow](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Multicol_layout/Handling_overflow) 的连续媒体溢出列；[Printing](https://developer.mozilla.org/en-US/docs/Web/CSS/Guides/Media_queries/Printing) 与 [@page 的 Page properties](https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@page#page_properties) 的打印规则和支持范围。
- Python 3.12：[http.server 命令行](https://docs.python.org/3.12/library/http.server.html#command-line-interface) 的本地静态服务。